In [ ]:
# 1. Install dependencies
!pip install transformers datasets accelerate llama-stack

# 2. Download Llama-3.1-8B directly from Meta
# PASTE YOUR UNIQUE URL BELOW:
meta_url = "YOUR_CUSTOM_URL_HERE"
!llama download --model-id Llama-3.1-8B --custom-url $meta_url

# 3. Convert Meta weights to Hugging Face format
!python -m transformers.models.llama.convert_llama_weights_to_hf \
    --input_dir ~/.llama/checkpoints/Llama3.1-8B \
    --model_size 8B \
    --output_dir ./Llama-3.1-8B-HF \
    --llama_version 3.1


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from src.ultrametric_v3.llama_patcher import inject_surgery
from src.ultrametric_v3.surgery_trainer import TauAnnealingCallback, SurgeryTrainer

model_id = "./Llama-3.1-8B-HF"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")

# Patch the model
model = inject_surgery(model)

# Freeze non-surgery parameters if desired (optional, but recommended for fine-tuning surgery routers)
for name, param in model.named_parameters():
    if "router" not in name:
        param.requires_grad = False

# Dummy dataset for demonstration
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

training_args = TrainingArguments(
    output_dir="./surgery_results",
    per_device_train_batch_size=2,
    learning_rate=1e-3,
    max_steps=100,
    logging_steps=10,
    save_steps=50,
)

trainer = SurgeryTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    callbacks=[TauAnnealingCallback(initial_tau=1.0, min_tau=0.1, decay_steps=100)],
    surgery_lambda_init=0.0,
    surgery_lambda_max=1.0,
    surgery_ramp_steps=50
)

trainer.train()
